# 05 — GAT GNN: Graph Attention Network (Blue Agent — Stream 2)

**Week 5 | RAKSHAK-ICS**

This notebook trains the **Graph Attention Network (SensorGAT)** — the spatial anomaly
detection stream of the Blue Agent. It captures inter-sensor dependencies through a
Pearson-correlation sensor graph with 65 nodes and 436 edges.

**Architecture:**  
- GATConv Layer 1: `(5 → 16, heads=8)` → 128-dim concat → ELU  
- GATConv Layer 2: `(128 → 5, heads=1)` → reconstruction head  
- Anomaly score: mean per-node MSE(input_features, reconstructed_features)

**Evaluation:** 5-seed training; F1/AUC-ROC mean±std; paired t-test vs. LSTM-AE baseline.

## 1. Setup & Imports

In [1]:
import sys
import logging
import json
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(message)s")

# Project root
ROOT = Path("..") if Path("../src").exists() else Path(".")
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.style as mstyle
mstyle.use("dark_background")
import torch
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score

from src.gnn import SensorGAT, train_gat, compute_gat_scores, tune_gat_threshold, GATBlueAgent
from src.stat_utils import set_all_seeds, paired_ttest, cohens_d, format_result

PROOF   = ROOT / "data" / "proof"
MODELS  = ROOT / "models"
FIGURES = ROOT / "results" / "figures"
TABLES  = ROOT / "results" / "tables"
MODELS.mkdir(exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 123, 456, 789, 1024]
ANOMALY_RATIO = 0.05
DEVICE = torch.device("cpu")   # change to "cuda" if GPU available

print(f"PyTorch: {torch.__version__}")
print(f"Using device: {DEVICE}")
try:
    import torch_geometric
    print(f"PyTorch Geometric: {torch_geometric.__version__}")
except ImportError:
    print("PyTorch Geometric: NOT available (using pure-PyTorch GAT fallback)")
print("Setup complete ✓")

2026-06-25 21:38:18,819 rakshak.gnn PyTorch Geometric NOT found — using pure-PyTorch GAT fallback


PyTorch: 2.1.2
Using device: cpu
PyTorch Geometric: NOT available (using pure-PyTorch GAT fallback)
Setup complete ✓


## 2. Load Preprocessed Graph Data

In [2]:
# Load preprocessed node features and graph topology
X_train_nf = np.load(PROOF / "node_features_train.npy")  # (N_train, 65, 5)
X_val_nf   = np.load(PROOF / "node_features_val.npy")    # (N_val,   65, 5)
X_test_nf  = np.load(PROOF / "node_features_test.npy")   # (N_test,  65, 5)
edge_index  = np.load(PROOF / "edge_index.npy")           # (2, E)
edge_weights = np.load(PROOF / "edge_weights.npy")        # (E,)

# Load LSTM windows for GATBlueAgent context
X_train = np.load(PROOF / "X_train.npy")  # (N_train, 60, 65)
X_val   = np.load(PROOF / "X_val.npy")    # (N_val,   60, 65)
X_test  = np.load(PROOF / "X_test.npy")   # (N_test,  60, 65)

N_NODES = X_train_nf.shape[1]  # 65
N_FEATS = X_train_nf.shape[2]  # 5
N_EDGES = edge_index.shape[1]

print(f"Train snapshots : {X_train_nf.shape}")
print(f"Val   snapshots : {X_val_nf.shape}")
print(f"Test  snapshots : {X_test_nf.shape}")
print(f"Graph           : {N_NODES} nodes, {N_EDGES} edges")
print(f"Edge weight range: [{edge_weights.min():.3f}, {edge_weights.max():.3f}]")

Train snapshots : (60966, 65, 5)
Val   snapshots : (13018, 65, 5)
Test  snapshots : (13019, 65, 5)
Graph           : 65 nodes, 436 edges
Edge weight range: [0.701, 1.000]


## 3. Synthetic Anomaly Injection (Test Set)

In [3]:
def inject_anomalies(X, ratio=0.05, seed=42):
    """Inject synthetic anomalies into the test set for evaluation."""
    rng = np.random.default_rng(seed)
    X_aug = X.copy()
    n = len(X_aug)
    n_anom = int(n * ratio)
    anom_idx = rng.choice(n, size=n_anom, replace=False)
    # Perturb: add Gaussian noise 3x std of each feature
    for idx in anom_idx:
        feature_idx = rng.integers(0, X_aug.shape[1])
        X_aug[idx, feature_idx, :] += rng.normal(0, 3.0 * X_aug[:, feature_idx, :].std())
    y = np.zeros(n, dtype=int)
    y[anom_idx] = 1
    return X_aug, y

X_test_nf_aug, y_test = inject_anomalies(X_test_nf, ratio=ANOMALY_RATIO, seed=42)
X_val_nf_aug,  y_val  = inject_anomalies(X_val_nf,  ratio=ANOMALY_RATIO, seed=99)

print(f"Test anomalies: {y_test.sum()} / {len(y_test)} ({y_test.mean()*100:.1f}%)")
print(f"Val  anomalies: {y_val.sum()} / {len(y_val)} ({y_val.mean()*100:.1f}%)")

Test anomalies: 650 / 13019 (5.0%)
Val  anomalies: 650 / 13018 (5.0%)


## 4. Single-Seed Training (Seed 42) — Loss Curves

In [4]:
set_all_seeds(42)

model_demo = SensorGAT(
    node_feature_dim=N_FEATS,
    hidden_dim=16,
    num_heads=8,
    dropout=0.1,
    edge_dim=1,
)

total_params = sum(p.numel() for p in model_demo.parameters())
print(f"SensorGAT parameters: {total_params:,}")
print(f"Using PyG: {model_demo.use_pyg}")

t_start = time.time()
history = train_gat(
    model_demo,
    X_train_nf,
    edge_index,
    edge_weights,
    X_val_nf,
    learning_rate=0.001,
    epochs=50,
    patience=15,
    subsample=3,   # take every 3rd snapshot for speed
    device=DEVICE,
)
elapsed = time.time() - t_start
print(f"\nTraining time (seed 42): {elapsed:.1f}s")

SensorGAT parameters: 1,945
Using PyG: False


2026-06-25 21:38:29,520 rakshak.gnn   Epoch   1/50  train=0.236811  val=0.216409  lr=1.00e-03  [9.8s/ep, 80 batches]


2026-06-25 21:39:06,527 rakshak.gnn   Epoch   5/50  train=0.115130  val=0.111187  lr=1.00e-03  [9.3s/ep, 80 batches]


2026-06-25 21:39:53,867 rakshak.gnn   Epoch  10/50  train=0.104788  val=0.106715  lr=1.00e-03  [9.9s/ep, 80 batches]


2026-06-25 21:40:42,925 rakshak.gnn   Epoch  15/50  train=0.104777  val=0.106730  lr=5.00e-04  [9.9s/ep, 80 batches]


2026-06-25 21:41:32,261 rakshak.gnn   Epoch  20/50  train=0.104770  val=0.106712  lr=5.00e-04  [9.9s/ep, 80 batches]


2026-06-25 21:41:51,987 rakshak.gnn   Early stopping at epoch 22 (patience=15)



Training time (seed 42): 212.4s


In [5]:
fig, ax = plt.subplots(figsize=(10, 4))
epochs_range = range(1, len(history['train_loss']) + 1)
ax.plot(epochs_range, history['train_loss'], label='Train Loss', color='#4FC3F7', lw=2)
ax.plot(epochs_range, history['val_loss'],   label='Val Loss',   color='#FF8A65', lw=2, ls='--')
ax.set_xlabel('Epoch', color='white')
ax.set_ylabel('MSE Reconstruction Loss', color='white')
ax.set_title('GAT GNN Training Loss Curves (Seed 42)', color='white', fontsize=14, pad=15)
ax.legend(facecolor='#1E1E1E', edgecolor='white')
ax.set_facecolor('#1E1E1E')
fig.patch.set_facecolor('#121212')
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_color('#555555')

plt.tight_layout()
out_path = FIGURES / "gat_loss.png"
plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='#121212')
plt.show()
print(f"Saved → {out_path}")

Saved → ../results/figures/gat_loss.png


## 5. Reconstruction Score Distribution & Threshold Tuning

In [6]:
# Compute val scores (normal-only)
val_scores_demo = compute_gat_scores(model_demo, X_val_nf, edge_index, edge_weights, device=DEVICE)
# Compute val scores on augmented (with anomalies)
val_scores_aug  = compute_gat_scores(model_demo, X_val_nf_aug, edge_index, edge_weights, device=DEVICE)

tau_demo = tune_gat_threshold(val_scores_demo, percentile=95.0)
print(f"Threshold τ = {tau_demo:.6f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution plot
normal_scores = val_scores_aug[y_val == 0]
anom_scores   = val_scores_aug[y_val == 1]
axes[0].hist(normal_scores, bins=60, alpha=0.7, color='#4FC3F7', label='Normal',   density=True)
axes[0].hist(anom_scores,   bins=60, alpha=0.7, color='#FF8A65', label='Anomaly',  density=True)
axes[0].axvline(tau_demo, color='#F06292', lw=2, ls='--', label=f'τ={tau_demo:.4f}')
axes[0].set_xlabel('GAT Reconstruction Score', color='white')
axes[0].set_ylabel('Density', color='white')
axes[0].set_title('Val Score Distribution', color='white')
axes[0].legend(facecolor='#1E1E1E')
axes[0].set_facecolor('#1E1E1E')
axes[0].tick_params(colors='white')

# Score timeline
axes[1].scatter(range(len(val_scores_aug)), val_scores_aug,
                c=['#FF8A65' if y == 1 else '#4FC3F7' for y in y_val],
                s=4, alpha=0.6)
axes[1].axhline(tau_demo, color='#F06292', lw=1.5, ls='--', label=f'τ={tau_demo:.4f}')
axes[1].set_xlabel('Window Index', color='white')
axes[1].set_ylabel('GAT Anomaly Score', color='white')
axes[1].set_title('Score Timeline (Val, blue=normal, orange=anomaly)', color='white')
axes[1].set_facecolor('#1E1E1E')
axes[1].tick_params(colors='white')

for ax in axes:
    ax.set_facecolor('#1E1E1E')
    for spine in ax.spines.values(): spine.set_color('#555555')
fig.patch.set_facecolor('#121212')
plt.tight_layout()
out_path = FIGURES / "gat_val_distribution.png"
plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='#121212')
plt.show()
print(f"Saved → {out_path}")

2026-06-25 21:41:56,343 rakshak.gnn   GAT threshold τ = 0.118214 (at 95.0th percentile of val)


Threshold τ = 0.118214


Saved → ../results/figures/gat_val_distribution.png


## 6. Multi-Seed Evaluation (5 Seeds)

In [7]:
all_results = []

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f"SEED {seed}")
    print('='*60)
    set_all_seeds(seed)

    model_s = SensorGAT(node_feature_dim=N_FEATS, hidden_dim=16, num_heads=8, dropout=0.1, edge_dim=1)

    history_s = train_gat(
        model_s, X_train_nf, edge_index, edge_weights, X_val_nf,
        learning_rate=0.001, epochs=50, patience=15, subsample=3, device=DEVICE,
    )

    # Threshold on normal val
    val_sc = compute_gat_scores(model_s, X_val_nf, edge_index, edge_weights, device=DEVICE)
    tau_s  = tune_gat_threshold(val_sc)

    # Evaluate on augmented test
    X_test_s, y_test_s = inject_anomalies(X_test_nf, ratio=ANOMALY_RATIO, seed=seed)
    test_sc = compute_gat_scores(model_s, X_test_s, edge_index, edge_weights, device=DEVICE)
    y_pred  = (test_sc > tau_s).astype(int)

    try:
        auc = roc_auc_score(y_test_s, test_sc)
    except ValueError:
        auc = 0.5

    metrics = {
        "seed":      seed,
        "f1":        float(f1_score(y_test_s, y_pred, zero_division=0)),
        "precision": float(precision_score(y_test_s, y_pred, zero_division=0)),
        "recall":    float(recall_score(y_test_s, y_pred, zero_division=0)),
        "auc_roc":   float(auc),
        "threshold": float(tau_s),
    }
    all_results.append(metrics)
    print(f"  F1={metrics['f1']:.4f}  P={metrics['precision']:.4f}  "
          f"R={metrics['recall']:.4f}  AUC={metrics['auc_roc']:.4f}")

    # Save best model (seed 42)
    if seed == 42:
        torch.save(model_s.state_dict(), MODELS / "gnn.pt")
        print(f"  Saved models/gnn.pt (seed 42)")

print("\n✓ 5-seed evaluation complete")


SEED 42


2026-06-25 21:42:06,588 rakshak.gnn   Epoch   1/50  train=0.236605  val=0.204567  lr=1.00e-03  [9.8s/ep, 80 batches]


2026-06-25 21:42:45,774 rakshak.gnn   Epoch   5/50  train=0.114236  val=0.110128  lr=1.00e-03  [9.8s/ep, 80 batches]


2026-06-25 21:43:34,888 rakshak.gnn   Epoch  10/50  train=0.104788  val=0.106713  lr=1.00e-03  [9.8s/ep, 80 batches]


2026-06-25 21:44:24,401 rakshak.gnn   Epoch  15/50  train=0.104775  val=0.106739  lr=5.00e-04  [9.9s/ep, 80 batches]


...

2026-06-25 21:59:02,110 rakshak.gnn   Epoch  15/50  train=0.104775  val=0.106719  lr=1.00e-03  [9.9s/ep, 80 batches]


2026-06-25 21:59:51,670 rakshak.gnn   Epoch  20/50  train=0.104777  val=0.106725  lr=5.00e-04  [9.8s/ep, 80 batches]


2026-06-25 22:00:21,306 rakshak.gnn   Early stopping at epoch 23 (patience=15)


2026-06-25 22:00:23,422 rakshak.gnn   GAT threshold τ = 0.118036 (at 95.0th percentile of val)


  F1=0.3378  P=0.3257  R=0.3508  AUC=0.7365

✓ 5-seed evaluation complete


## 7. Results Summary (mean ± std)

In [8]:
metric_keys = ["f1", "precision", "recall", "auc_roc"]
aggregated = {}
for k in metric_keys:
    vals = [r[k] for r in all_results]
    aggregated[k] = {
        "scores": vals, "mean": np.mean(vals), "std": np.std(vals),
        "formatted": f"{np.mean(vals):.3f}±{np.std(vals):.3f}"
    }
print("=" * 55)
print(f"{'GAT GNN — 5-Seed Results (SWaT A9)':^55}")
print("=" * 55)
print(f"{'Metric':<18} {'Mean':>10} {'Std':>10} {'Formatted':>15}")
print("-" * 55)
for k, v in aggregated.items():
    print(f"{k:<18} {v['mean']:>10.4f} {v['std']:>10.4f} {v['formatted']:>15}")
print("=" * 55)
# Compare with LSTM-AE
try:
    with open(TABLES / "lstm_ae_results.json") as f:
        lstm_res = json.load(f)
    lstm_f1 = lstm_res["f1"]["scores"]
    gat_f1  = [r["f1"] for r in all_results]
    stat_test = paired_ttest(gat_f1, lstm_f1)
    t_stat, p_val = stat_test["t_statistic"], stat_test["p_value"]
    d = cohens_d(gat_f1, lstm_f1)
    print(f"\nPaired t-test (GAT vs LSTM-AE):")
    print(f"  t = {t_stat:.4f},  p = {p_val:.4e},  Cohen's d = {d:.4f}")
    sig = "SIGNIFICANT" if p_val < 0.05 else "not significant"
    print(f"  → {sig} at α=0.05")
except FileNotFoundError:
    print("(LSTM-AE results not found, skipping t-test)")
# Save results
import json as _json
gat_results = {"per_seed": all_results, "aggregated": {
    k: {"scores": v["scores"], "mean": float(v["mean"]), "std": float(v["std"]),
        "formatted": v["formatted"]}
    for k, v in aggregated.items()
}}
with open(TABLES / "gat_results.json", "w") as f:
    _json.dump(gat_results, f, indent=2)
print(f"\nSaved → results/tables/gat_results.json")


          GAT GNN — 5-Seed Results (SWaT A9)           
Metric                   Mean        Std       Formatted
-------------------------------------------------------
f1                     0.3250     0.0198     0.325±0.020
precision              0.3172     0.0186     0.317±0.019
recall                 0.3332     0.0214     0.333±0.021
auc_roc                0.7305     0.0084     0.730±0.008

Paired t-test (GAT vs LSTM-AE):
  t = 29.9047,  p = 7.4467e-06,  Cohen's d = 19.0406
  → SIGNIFICANT at α=0.05

Saved → results/tables/gat_results.json


## 8. Visualisations — F1 per Seed & Sensor Attribution

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F1 per seed
f1_vals = [r["f1"] for r in all_results]
colors  = ['#4FC3F7', '#FF8A65', '#A5D6A7', '#CE93D8', '#FFF176']
bars = axes[0].bar([str(s) for s in SEEDS], f1_vals, color=colors, edgecolor='white', linewidth=0.5)
axes[0].axhline(np.mean(f1_vals), color='#F06292', ls='--', lw=2, label=f'Mean={np.mean(f1_vals):.3f}')
axes[0].set_xlabel('Seed', color='white')
axes[0].set_ylabel('F1 Score', color='white')
axes[0].set_title('GAT GNN — F1 per Seed', color='white', fontsize=13)
axes[0].legend(facecolor='#1E1E1E')
axes[0].set_facecolor('#1E1E1E')
axes[0].tick_params(colors='white')
for bar, val in zip(bars, f1_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                 f'{val:.3f}', ha='center', va='bottom', color='white', fontsize=9)

# Metric comparison
metric_means = [aggregated[k]["mean"] for k in metric_keys]
metric_stds  = [aggregated[k]["std"]  for k in metric_keys]
metric_labels = ["F1", "Precision", "Recall", "AUC-ROC"]
bar_colors = ['#4FC3F7', '#FF8A65', '#A5D6A7', '#CE93D8']
bars2 = axes[1].bar(metric_labels, metric_means, yerr=metric_stds,
                    color=bar_colors, edgecolor='white', capsize=5, linewidth=0.5)
axes[1].set_ylabel('Score', color='white')
axes[1].set_title('GAT GNN — Metrics (mean ± std)', color='white', fontsize=13)
axes[1].set_facecolor('#1E1E1E')
axes[1].tick_params(colors='white')
axes[1].set_ylim(0, max(metric_means) * 1.4)
for bar, val, err in zip(bars2, metric_means, metric_stds):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + err + 0.002,
                 f'{val:.3f}', ha='center', va='bottom', color='white', fontsize=9)

for ax in axes:
    ax.set_facecolor('#1E1E1E')
    for spine in ax.spines.values(): spine.set_color('#555555')
fig.patch.set_facecolor('#121212')
plt.tight_layout()
out_path = FIGURES / "gat_f1_metrics.png"
plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='#121212')
plt.show()
print(f"Saved → {out_path}")

2026-06-25 22:00:25,856 matplotlib.category Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


2026-06-25 22:00:25,857 matplotlib.category Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


Saved → ../results/figures/gat_f1_metrics.png


## 9. Summary

### SensorGAT Architecture
- **Layer 1**: GATConv(5→16, heads=8) → 128-dim concat → ELU + LayerNorm + Dropout(0.1)  
- **Layer 2**: GATConv(128→5, heads=1) → reconstruction head → LayerNorm  
- **Anomaly score**: Mean per-node MSE over 65 sensors

### Key Findings
- Reconstruction-based GAT captures spatial inter-sensor dependencies
- All scores trained on **normal-only** data (unsupervised anomaly detection)
- Threshold τ at **95th percentile** of normal validation scores
- Results form the **baseline for ablation** comparison in `05b_ablation.ipynb`

### Files Saved
- `models/gnn.pt` — Best GAT weights (seed 42)
- `results/figures/gat_loss.png` — Training loss curves
- `results/figures/gat_val_distribution.png` — Score distributions
- `results/figures/gat_f1_metrics.png` — Metric bar charts
- `results/tables/gat_results.json` — All 5-seed metrics

**Next**: `05b_ablation.ipynb` — 4-config ablation study (LSTM-only / GAT-only / no-fusion / full-fused)
